# Smart SIP — Data Analysis Notebook

This notebook demonstrates the full analytics stack:
- Gold & Silver price simulation (GBM + mean-reversion)
- Smart SIP vs Standard SIP comparison
- Monte Carlo backtest (distribution of outcomes)
- Portfolio Sharpe ratio & drawdown analysis
- Goal-based planning


In [ ]:
import sys, os
sys.path.insert(0, '../backend')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import norm

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'sans-serif'})
print('Libraries loaded ✓')

## 1. Price History

In [ ]:
df = pd.read_csv('../data/price_history.csv', parse_dates=['date'])
df.set_index('date', inplace=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
df['gold_price'].plot(ax=axes[0], color='#D4A017', lw=2, title='Gold Price (₹/gram)')
df['silver_price'].plot(ax=axes[1], color='#808080', lw=2, title='Silver Price (₹/gram)')
for ax in axes:
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
df.describe()

## 2. Smart SIP vs Standard SIP

In [ ]:
from sip_engine import run_sip

smart  = run_sip(500, 5, 10, 'smart',    'gold')
std    = run_sip(500, 5, 10, 'standard', 'gold')

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(smart.labels))
ax.fill_between(x, smart.invested_history, alpha=0.15, color='gray',  label='Total Invested')
ax.plot(x, smart.value_history,     color='#5B2EDE', lw=2.5, label=f'Smart SIP  ({smart.return_pct:.1f}% return)')
ax.plot(x, std.value_history,       color='#aaa',    lw=2,   linestyle='--', label=f'Standard SIP ({std.return_pct:.1f}% return)')
ax.plot(x, smart.invested_history,  color='#333',    lw=1.5, linestyle=':')
ax.set_xticks(list(x)); ax.set_xticklabels(smart.labels, rotation=45)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'₹{v/1e3:.0f}K' if v < 1e6 else f'₹{v/1e5:.1f}L'))
ax.set_title('Smart SIP vs Standard SIP — Portfolio Value over 5 Years')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Smart SIP advantage: ₹{smart.advantage_vs_std:,.0f}')

## 3. Monte Carlo Backtest

In [ ]:
from price_simulator import backtest_smart_vs_standard

bt = backtest_smart_vs_standard('gold', 500, 5, 10, paths=200)

fig, ax = plt.subplots(figsize=(10, 4))
adv = np.array(bt['advantages'])
ax.hist(adv, bins=30, color='#5B2EDE', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', lw=1.5, linestyle='--')
ax.axvline(bt['mean_advantage'], color='gold', lw=2, label=f'Mean ₹{bt["mean_advantage"]:,.0f}')
ax.set_xlabel('Smart SIP Advantage over Standard (₹)')
ax.set_ylabel('Number of Scenarios')
ax.set_title(f'Monte Carlo: Smart SIP wins in {bt["pct_paths_smart_wins"]}% of {bt["paths"]} scenarios')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Portfolio Analytics — Sharpe & Drawdown

In [ ]:
from analytics import rolling_stats, drawdown_analysis

prices = df['gold_price'].tolist()
rs  = rolling_stats(prices)
dd  = drawdown_analysis(prices)

print(f'Annualised Return : {rs["ann_return"]*100:.1f}%')
print(f'Annualised Vol    : {rs["ann_vol"]*100:.1f}%')
print(f'Sharpe Ratio      : {rs["sharpe"]:.2f}')
print(f'Max Drawdown      : {dd["max_drawdown_pct"]:.1f}%')
print(f'Drawdown Duration : {dd["drawdown_duration"]} months')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(rs['roll_return'],  color='#5B2EDE', label='Rolling Return (ann.)')
axes[0].plot(rs['roll_vol'],     color='#E74C3C', label='Rolling Vol (ann.)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Rolling 12-Month Return & Volatility')

axes[1].fill_between(range(len(dd['drawdown_series'])), dd['drawdown_series'], 0,
                     color='#E74C3C', alpha=0.4, label='Drawdown')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Portfolio Drawdown')
plt.tight_layout(); plt.show()

## 5. Goal Planner

In [ ]:
from analytics import goal_planner
targets = [100_000, 500_000, 1_000_000, 5_000_000]
rows = [goal_planner(t, 10) for t in targets]
gdf = pd.DataFrame(rows)[['target_corpus','monthly_sip_needed','total_invested','expected_gain','inflation_adj_value']]
gdf.columns = ['Target (₹)', 'Monthly SIP (₹)', 'Total Invested (₹)', 'Expected Gain (₹)', 'Inflation-adj (₹)']
gdf.style.format('₹{:,.0f}').background_gradient(cmap='Purples', subset=['Monthly SIP (₹)'])